[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/affine_toda/affine_toda.ipynb)

# E8 Toda Masses and Interactions

Affine E8 Toda theory describes eight coupled scalar fields in one dimension of space and one of time. The curvature of their potential at equilibrium gives eight particle masses, and its cubic term decides which groups of three particles interact.

The field takes values in the eight-dimensional space of E8's roots, antivectors, and each root, a mirror, a vector, reads it by pairing, `root & field`. The potential is a weighted sum of exponentials of those readings, so its curvature and its cubic term at the vacuum are forms on the field, sums of products of the readings. The roots and the plane they are drawn in come from [E8 Reflections](../../math/e8/e8.ipynb).

The masses turn out to be the radii of the rings in the E8 picture, and three particles interact exactly when roots from their three rings close into a triangle.

In index notation, the potential with the marks $n_i$ and the negative highest root $\alpha_0$ reads $V(\phi) = \frac{m^2}{\beta^2}\sum_{i=0}^{8} n_i \left(e^{\beta\,\alpha_i\cdot\phi} - 1\right)$, and its expansion about $\phi = 0$ gives the mass matrix $M^2_{ab} = m^2 \sum_i n_i\, \alpha_i^a \alpha_i^b$ and the cubic couplings $C_{abc} = m^2 \beta \sum_i n_i\, \alpha_i^a \alpha_i^b \alpha_i^c$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import concatenate, stack
from examples.util.animation import save_figure
from examples.math.e8 import core as e8
from examples.math.e8.scenarios import root_system
from examples.quantum.affine_toda import render

In [ ]:
ga = e8.ga
# The field's values are antivectors; the roots that read them are mirrors, vectors.
Field = ga.gatype.antivector()                                   # Antivector-

root_orbits, plane_x, plane_y, planes, *_ = root_system()        # [steps, orbits] Root+, [planes] Root+, [planes] Root+, [planes] Bivector+
horizontal = plane_x[0]                                          # [] Root+
vertical = plane_y[0]                                            # [] Root+
coxeter_plane = planes[0]                                        # [] Bivector+

## 1. The vacuum curvature

The potential sums exponentials of the field read by nine roots: the eight simple roots, and the negative of the highest root, weighted so that the nine sum to zero, which makes zero field an equilibrium. Each exponential's second derivative at zero is the product of its root's readings, so the curvature is `(marks * reading * reading).sum()`, a form on two fields. With a canonical kinetic term its eigenvalues against the field's metric are the squared masses. Both the overall mass scale and the coupling in the exponent are set to one.

In [ ]:
# The highest root's coefficients on the simple roots, followed by the weight of its negative.
marks = np.array([2, 3, 4, 6, 5, 4, 3, 2, 1])

In [ ]:
simple_roots = e8.simple_roots()                                 # [simple_roots] Root+
affine_root = -(marks[:-1] * simple_roots).sum(axis=0)           # [] Root+
interaction_roots = concatenate((simple_roots, affine_root[None]))   # [interactions] Root+

# Each root reads one component of the field.
reading = interaction_roots & Field                              # [interactions] Scalar+ <- Field-
curvature = (marks * reading * reading).sum(axis=0)              # [] Scalar+ <- (Field-, Field-)
eigenvalues, modes = curvature.eigh()                            # [species] Scalar+, Field-
masses = eigenvalues.square_root()                               # [species] Scalar+

# Pair the eight orbits with the modes by increasing radius and increasing mass.
radii = (root_orbits[0] | coxeter_plane).scalar_norm_squared().square_root()   # [orbits] Scalar+
order = radii.argsort()
root_orbits = root_orbits[:, order]                              # [steps, species] Root+
radii = radii[order]                                             # [species] Scalar+

In [ ]:
print("masses over the lightest:", (masses / masses[0]).to_array())
print("radii over the smallest:", (radii / radii[0]).to_array())
figure = render.draw_roots(root_orbits, horizontal, vertical)
save_figure(figure, "affine_toda_masses")
figure

The mass ratios equal the ratios of the rings' radii. Colours mark the particles in increasing mass: yellow, red, pink, violet, blue, cyan, green and lime. The dots are the roots, and faint edges join nearest neighbours in the full eight-dimensional space. The picture is built entirely from reflections, yet its radii are a spectrum of excitations; [Koca and Koca](https://arxiv.org/abs/1204.4567) describe this correspondence.

## 2. Measured excitation energies

Neutron scattering measures the energy needed to create a spin excitation, and at the zone centre the E8 Ising scaling theory predicts the same energy ratios as the Toda masses. [Wang and colleagues, Table I](https://arxiv.org/pdf/2308.00249#page=3), report three peak energies measured on IN12 in BaCo₂V₂O₈ at a transverse field of 4.7 T, a temperature of 1.5 K and wavevector $(0,0,2)$. The first peak sets the energy scale; the other two follow from the mass ratios.

In [ ]:
# IN12 peak positions in meV, from Gaussian fits to the neutron-scattering data.
measured_energies = np.array([1.26, 2.05, 2.49])
n_measured_modes = len(measured_energies)

In [ ]:
# Only the overall scale comes from experiment; the ratios come from the curvature.
predicted_energies = measured_energies[0] * masses[:n_measured_modes] / masses[0]   # [measured_modes] Scalar+

In [ ]:
print("E8 predictions, meV:", predicted_energies.to_array())
print("measured peaks, meV:", measured_energies)

The second and third predicted energies lie close to the measured peaks. The source gives peak positions without fitted uncertainties, and the third peak overlaps a two-particle threshold. The comparison concerns the shared E8 mass spectrum; the cubic couplings below stay calculations within classical Toda theory.

## 3. Three-mode couplings

The third derivative of the same potential is a form on three fields, `(marks * reading * reading * reading).sum()`, and applying it to three modes gives their cubic coupling. A coupling is nonzero only when roots from the three particles' rings sum to zero. The examples are the triples 1–1–2, 1–2–3 and 2–3–5; the last does not couple, though its three masses satisfy the ordinary triangle inequalities.

In [ ]:
species = np.array([[0, 0, 1], [0, 1, 2], [1, 2, 4]])
n_cases = len(species)
n_steps = root_orbits.shape[0]
steps = np.arange(n_steps)
cases = np.arange(n_cases)
closure_tolerance = 1e-10

In [ ]:
cubic = (marks * reading * reading * reading).sum(axis=0)        # [] Scalar+ <- (Field-, Field-, Field-)
couplings = cubic(modes[species[:, 0]], modes[species[:, 1]], modes[species[:, 2]])   # [cases] Scalar+

# A common Coxeter turn preserves closure, so the first root can stay at step zero.
first_roots = root_orbits[0, species[:, 0]]                      # [cases] Root+
second_roots = root_orbits[steps[None, :], species[:, 1, None]]  # [cases, steps] Root+
third_roots = root_orbits[steps[None, :], species[:, 2, None]]   # [cases, steps] Root+

# Search the remaining two rings in the full eight-dimensional space.
residuals = first_roots[:, None, None] + second_roots[:, :, None] + third_roots[:, None, :]   # [cases, steps, steps] Root+
closure_squared = residuals.scalar_norm_squared().reshape(n_cases, -1)   # [cases, step_pairs] Scalar+
nearest = closure_squared.argmin(axis=-1)
triangles = stack((first_roots, second_roots[cases, nearest // n_steps],
                   third_roots[cases, nearest % n_steps]), axis=1)   # [cases, sides] Root+
closure = closure_squared[cases, nearest]                        # [cases] Scalar+

# Joining the roots head to tail shows whether their sum returns to the origin.
paths = concatenate((0 * triangles[:, :1], triangles.cumsum(axis=1)), axis=1)   # [cases, corners] Root+

In [ ]:
print("species:", species + 1)
print("|coupling|:", couplings.abs().to_array())
print("root closure:", closure.square_root().to_array())
figure = render.draw_triangles(paths, root_orbits, species, horizontal, vertical)
save_figure(figure, "affine_toda_triangles")
figure

The coloured arrows are roots placed head to tail, from a white dot. The first two paths close; the third leaves a dashed gap even for the closest choice of roots, matching its zero coupling. Closure is tested in eight dimensions and the drawing is its projection: the discrete directions of the roots impose a stronger condition than their lengths alone.

## 4. Coupling strength from area

For a closed root triangle, its area in the Coxeter plane gives the coupling's magnitude. The table computes that magnitude twice within the same classical Toda model, from the cubic form and from the mass-scaled triangle area; neither column is a measurement.

In [ ]:
mass_scale = masses[0] / radii[0]                                # [] Scalar+
closed = closure < closure_tolerance
mass_triangles = triangles[closed] * mass_scale                  # [closed_cases, sides] Root+
# Pairing with the drawing plane measures the projected wedge area.
areas = (mass_triangles[:, 0] ^ mass_triangles[:, 1]).scalar_product(coxeter_plane).abs() / 2   # [closed_cases] Scalar+
predicted = 4 * areas / np.sqrt(e8.coxeter_order)                # [closed_cases] Scalar+

In [ ]:
print("from the cubic form:", couplings[closed].abs().to_array())
print("from triangle area:", predicted.to_array())

In [ ]:
# checks
np.testing.assert_allclose((radii / radii[0] - masses / masses[0]).to_array(), 0, atol=1e-10)
np.testing.assert_array_equal(closed, couplings.abs() > 1e-10)
np.testing.assert_allclose((couplings[closed].abs() - predicted).to_array(), 0, atol=1e-10)

The curvature, the cubic form and the triangle areas are three views of one interaction geometry. [Dorey's root-triangle rule](https://arxiv.org/abs/hep-th/9110058) relates the nonzero couplings and their magnitudes to the Coxeter orbits.